# first letter and concat words together

In [ ]:
-- LEFT is able to get the first letter from the left of the word
SELECT CONCAT(Name, '(', LEFT(Occupation,1), ')')
FROM OCCUPATIONS
ORDER BY Name;

SELECT CONCAT(
    'There are a total of ',
    COUNT(*),
    ' ',
    LOWER(Occupation),
    's.'
)
FROM OCCUPATIONS
GROUP BY Occupation
ORDER BY COUNT(*), Occupation;

# practicing Pivot with Case

In [ ]:
SELECT
-- using max to get the value that is not null
    MAX(CASE WHEN Occupation = 'Doctor' THEN Name END) AS Doctor,
    MAX(CASE WHEN Occupation = 'Professor' THEN Name END) AS Professor,
    MAX(CASE WHEN Occupation = 'Singer' THEN Name END) AS Singer,
    MAX(CASE WHEN Occupation = 'Actor' THEN Name END) AS Actor
FROM (
    SELECT
        Name,
        Occupation,
        ROW_NUMBER() OVER (
            PARTITION BY Occupation
            ORDER BY Name
        ) AS rn
    FROM OCCUPATIONS
) t
GROUP BY rn
ORDER BY rn;

# BST problem, finding leaf, root, inner

In [ ]:
SELECT n,
    CASE 
    WHEN p IS NULL THEN 'Root'
    WHEN n NOT IN (SELECT p FROM BST) THEN 'Leaf'
    ELSE 'Inner'
    END 
    FROM BST
    ORDER BY n



# getting count for each employee

In [ ]:
SELECT c.company_code, c.founder, COUNT(DISTINCT e.lead_manager_code), COUNT(DISTINCT e.senior_manager_code), COUNT(DISTINCT e.manager_code), COUNT(DISTINCT e.employee_code)
FROM Company AS c
INNER JOIN Employee AS e
    ON c.company_code = e.company_code
    GROUP BY c.company_code, c.founder
    ORDER BY c.company_code;

    

# getting the manhattan distance (round number)

In [ ]:
SELECT ROUND(
    ABS(MAX(LAT_N) - MIN(LAT_N)) +
    ABS(MAX(LAT_W) - MIN(LAT_N)), 4
    ) FROM STATION

# finding the median

In [ ]:
SELECT
    ROUND(AVG(LAT_N), 4)
FROM (
    SELECT
        LAT_N,
        ROW_NUMBER() OVER (ORDER BY LAT_N) AS rn,
        COUNT(*) OVER () AS total
    FROM STATION
) t
WHERE rn IN (
    (total + 1) / 2,
    (total + 2) / 2
);

# finding the marks for each student

In [ ]:

SELECT 
CASE WHEN
g.grade >= 8 THEN s.name
ELSE NULL
END, g.grade, s.marks
FROM STUDENTS AS s
JOIN GRADES AS g
ON s.marks BETWEEN g.min_mark AND g.max_mark
ORDER BY
    CASE WHEN g.grade >= 8 THEN g.grade END DESC,
    CASE WHEN g.grade >= 8 THEN s.name END ASC,
    CASE WHEN g.grade < 8 THEN g.grade END DESC,
    CASE WHEN g.grade < 8 THEN s.marks END ASC;

# getting the top hackers with more than 1 perfect score

In [ ]:
SELECT h.hacker_id, h.hacker_name FROM Hackers AS h
JOIN Submisions AS s ON h.hacker_id = s.hacker_id
JOIN Challenges AS c  ON c.challenge_id = s.challenge_id
JOIN Difficulty AS d ON d.difficulty_level = c.dificulty_level
WHERE d.score = s.score
GROUP BY h.hacker_id, h.hacker_name
HAVING COUNT(*) > 1
ORDER BY 
COUNT(*) DESC, h.hacker_id ASC;


# getting the best priced Harry Potter Wand

In [ ]:
WITH ranked AS (
    SELECT
        w.id,
        wp.age,
        w.coins_needed,
        w.power,
        ROW_NUMBER() OVER (
            PARTITION BY w.power, wp.age
            ORDER BY w.coins_needed
        ) AS rn
    FROM Wands AS w
    JOIN Wands_property AS wp
        ON wp.code = w.code
    WHERE w.is_evil = 0
)

SELECT
    id,
    age,
    coins_needed,
    power
FROM ranked
WHERE rn = 1
ORDER BY
    power DESC,
    age DESC;

# get the hacker_id, name, and total number of challenges created by each student

In [ ]:
WITH challenge_counts AS (
    SELECT
        h.hacker_id,
        h.name,
        COUNT(c.challenge_id) AS total_challenges
    FROM Hackers h
    JOIN Challenges c
        ON h.hacker_id = c.hacker_id
    GROUP BY
        h.hacker_id,
        h.name
),

duplicates AS (
    SELECT
        total_challenges
    FROM challenge_counts
    GROUP BY total_challenges
    HAVING COUNT(*) > 1
)

SELECT
    hacker_id,
    name,
    total_challenges
FROM challenge_counts
WHERE total_challenges NOT IN (
    SELECT total_challenges
    FROM duplicates
)
OR total_challenges = (
    SELECT MAX(total_challenges)
    FROM challenge_counts
)
ORDER BY
    total_challenges DESC,
    hacker_id ASC;

# add scores of challenges together (taking max score)

In [ ]:
WITH max_scores AS (
    SELECT
        hacker_id,
        challenge_id,
        MAX(score) AS max_score
    FROM Submissions
    GROUP BY
        hacker_id,
        challenge_id
)

SELECT
    h.hacker_id,
    h.name,
    SUM(ms.max_score) AS total_score
FROM Hackers h
JOIN max_scores ms
    ON h.hacker_id = ms.hacker_id
GROUP BY
    h.hacker_id,
    h.name
HAVING SUM(ms.max_score) > 0
ORDER BY
    total_score DESC,
    h.hacker_id ASC;


# getting consecutive dataes

In [ ]:
WITH marked AS (
    SELECT
        Start_Date,
        End_Date,
        CASE
            WHEN LAG(End_Date) OVER (ORDER BY Start_Date) = Start_Date
            THEN 0
            ELSE 1
        END AS new_project
    FROM Projects
),
grouped AS (
    SELECT
        Start_Date,
        End_Date,
        SUM(new_project) OVER (ORDER BY Start_Date) AS project_id
    FROM marked
)

SELECT
    MIN(Start_Date) AS project_start,
    MAX(End_Date) AS project_end
FROM grouped
GROUP BY project_id
ORDER BY
    DATEDIFF(MAX(End_Date), MIN(Start_Date)),
    project_start;

# comparing salary of friends and students

In [ ]:
SELECT 
s.name
FROM Students s
JOIN Friends f ON s.ID = f.ID
JOIN Packages p ON s.ID = p.ID
JOIN Packages p2 ON f.friend_id = p2.ID
WHERE p.salary<p2.salary
ORDER BY p2.salary ASC

# get the symetric pairs

In [ ]:
-- need row count

SELECT f.X, f.Y
FROM Functions f
JOIN Functions f2 ON f.X=f2.Y AND f.Y = f2.X
WHERE f.X < f.Y
OR (f.X = f.Y AND (SELECT COUNT(*) FROM functions f3 WHERE f3.x = f.x AND f3.y = f.y)>1)
ORDER BY f.X 

# LEVEL hard, finding the totals of total_subimissions, total_accepted_submissionsm total_views, total_unique_views

In [ ]:


WITH Contest_totals AS (
    SELECT con.contest_id, con.hacker_id, con.name, 
SUM(COALESCE(ss.total_submissions, 0)) AS tot_sub, 
SUM(COALESCE(ss.total_accepted_submissions, 0)) AS tot_acc_sub, 
SUM(COALESCE(vs.total_views, 0)) AS tot_vie, 
SUM(COALESCE(vs.total_unique_views, 0)) AS tot_uniq_view
FROM Contests con JOIN colleges col ON con.contest_id = col.contest_id
JOIN Challenges cha ON cha.college_id = col.college_id
LEFT JOIN Submission_stats ss ON cha.challenge_id = ss.challenge_id
LEFT JOIN View_stats vs ON cha.challenge_id = vs.challenge_id
GROUP BY con.contest_id, con.hacker_id, con.name
)
SELECT * FROM Contest_totals WHERE
tot_sub != 0 OR tot_acc_sub != 0 OR tot_vie != 0 OR tot_uniq_view != 0
ORDER BY 


In [ ]:
WITH rank AS (SELECT employee_id, RANK() OVER (
    PARTITION BY department
    ORDER BY salary
) AS rnk
FROM Employee
)

SELECT employee_id FROM rank WHERE rnk = 1



In [ ]:
WITH lag AS (
    SELECT user_id, login_date, LAG() OVER(
        PARTITION BY user_id
        ORDER BY login_date
    ) AS prev FROM user_login
),
running_total AS(
    SELECT user_id, login_date, 
    CASE WHEN DAY(login_date) - DAY(prev) = 1 THEN 
    1
    ELSE 0
    END AS running
    FROM lag
)

SELECT user_id FROM running_total 
GROUP BY user_id,  HAVING sum(running) >= 3

In [ ]:
WITH runnning AS(
    SELECT account_id, transaction_date, 
    SUM() OVER(
        PARTITION BY account_id
        ORDER BY transaction_date
        BETWEEN first_transaction AND transaction_date
    ) AS running_balance
)

In [ ]:
SELECT s.name
FROM Students AS s
JOIN Friends AS f ON s.id = f.id
JOIN Packages AS p ON s.id = p.id
JOIN Packages AS pf ON f.Friend_id = pf.id
WHERE pf.salary>p.salary
ORDER BY pf.salary

In [ ]:
-- gets the row number with the lat_n and the sum of the rows
WITH row_num AS (
    SELECT lat_n, 
    ROW_NUMBER() OVER(
        ORDER BY lat_n
    ) AS rn,
    COUNT(*) OVER() AS count
     FROM STATION
)

SELECT 
CASE WHEN count % 2 = 0 THEN
    ROUND(SELECT(lat_n FROM row_num WHERE rn = count/2) + SELECT(lat_n FROM row_num WHERE rn = (count/2)+1)/2, 4)
ELSE
    ROUND(SELECT lat_n FROM row_num WHERE rn = (count+1) / 2, 4)
END
FROM row_num
LIMIT 1;
    



In [ ]:
WITH top AS (
    SELECT employee_id, esmployee_name, department, salary, 
    RANK() OVER( 
        PARTITION BY department
        ORDER BY salary
    ) AS rank
    FROM EMPLOYEE 
)


SELECT 
employee_id, employee_name FROM top WHERE rank = 1